# S5.G3-1 · G3 五臂对比 FedAds：数据、调参与本地臂

> 按冻结的 S6.P3 与 S6.P4（DR-M6-002）执行。数据：FedAds 官方纵向切分（公开数据），主动方 `l_*` 对应内地、被动方 `f_*` 对应香港。
> 行数据、用户号、编码表与逐行打分只在维护者本机或会话私有目录，本 notebook 只输出汇总。不代表汇丰真实效果；技术侧自评，未经法务复核（W-001）。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_g3_fedads_run.yaml"
run = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
PRE_PATH, ADD_PATH = REPO_ROOT / run["preregistration"], REPO_ROOT / run["addendum"]
pre = yaml.safe_load(PRE_PATH.read_text(encoding="utf-8"))
add = yaml.safe_load(ADD_PATH.read_text(encoding="utf-8"))
ad, fa = pre["freeze"]["arm_details"], add["arm_details"]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("preregistration:", PRE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(PRE_PATH), "| status:", pre["status"])
print("addendum:", ADD_PATH.relative_to(REPO_ROOT), "sha256:" + sha(ADD_PATH), "| status:", add["status"])
print("seed:", run["seed"], "| seeds:", run["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", run["step_id"], "|", run["purpose"])
mismatch = [p for p, h in add["code_sha256"].items() if hashlib.sha256((REPO_ROOT / p).read_bytes()).hexdigest() != h]
print("冻结代码指纹核对：", "一致" if not mismatch else "不一致 " + str(mismatch))
assert pre["status"] == "frozen" and add["status"] == "frozen" and not mismatch
assert sha(PRE_PATH, None) == add["parent_sha256"]
PRIV = pathlib.Path(os.environ[run["private_root_env"]])
SEED0 = run["seeds"][0]
MB, NJ = ad["xgboost_max_bin"], ad["n_jobs"]

config: modules/m5_modeling/configs/s5_g3_fedads_run.yaml sha256:56f1fdbd6540
preregistration: modules/m6_evaluation/configs/s6_p3_g3_preregistration.yaml sha256:c0a4bbfc9709 | status: frozen
addendum: modules/m6_evaluation/configs/s6_p4_fedads_addendum.yaml sha256:462b0a74e10b | status: frozen
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: 540d5b9（工作区含本步未提交改动）
step: S5.G3 | 按冻结的 S6.P3 与 S6.P4 在 FedAds 上训练 L0、L1（k=5/10/20）、B_T1、A（SGB）、集中式代理 L4 与阴性对照 NC，并做配对比较
冻结代码指纹核对： 一致


In [2]:
from modules.m5_modeling.components import g3_fedads as F

prep = add["fedads_prep"]
src = PRIV / run["selected_rows_file"]
assert hashlib.sha256(src.read_bytes()).hexdigest() == prep["selected_rows_sha256"]
rows = F.parse_selected(str(src))
GROUPS = ("train", "inner_val", "dev_eval")
parts = {g: rows[rows[F.GROUP_COL] == g].reset_index(drop=True) for g in GROUPS}
F.validate_groups(prep["groups"])
users = {g: set(d[F.USER_COL]) for g, d in parts.items()}
overlap = {(a, b): len(users[a] & users[b]) for a in GROUPS for b in GROUPS if a < b}
print("各组用户重叠数（应全为 0）：", overlap)
assert not any(overlap.values())
counts = pd.DataFrame(F.summary_counts(list(parts.items())))
print(counts.to_string(index=False))
print("开发评估集正例行 %d（预注册目标 %d）"
      % (int(parts["dev_eval"][F.LABEL_COL].sum()), prep["eval_bucket_rule"]["target_positives"]))

各组用户重叠数（应全为 0）： {('inner_val', 'train'): 0, ('dev_eval', 'train'): 0, ('dev_eval', 'inner_val'): 0}
    group   rows  users  positive_rows  positive_users
    train 151281 103538            878             866
inner_val  59370  40851            391             389
 dev_eval 129489  91967            848             831
开发评估集正例行 848（预注册目标 700）


**读法**：三组用户两两不相交（重叠数全为 0）。开发评估集 129,489 行、91,967 位用户、848 个正例行，高于预注册目标 700；同一用户多次转化极少（848 行对应 831 位用户）。训练组正例 878 行，比 Santander 第一批（训练月约 3.8%）稀疏得多，绝对 AUROC 会明显偏低。

In [3]:
from modules.m5_modeling.components import g3_arms as A

enc = F.fit_encoder(parts["train"], fa["high_card_threshold"])
X = {}
for n, df in parts.items():
    xl, xf = F.encode(df, enc)
    X[n] = {"xg": xl, "xh": xf, "oh": F.follower_one_hot(df, enc), "y": df[F.LABEL_COL].to_numpy(),
            "cust": df[F.USER_COL].astype(np.int64).to_numpy()}
np.savez_compressed(PRIV / run["arrays_file"],
                    **{n + "__" + k: v for n, d in X.items() for k, v in d.items() if k != "oh"})
print("编码：", {c: k for c, (k, _) in enc.items() if k == "freq"}, "其余序号编码")
print({n: (d["xg"].shape, d["xh"].shape, d["oh"].shape) for n, d in X.items()})

# —— 各臂的内地侧特征集 ——
seg_cols = fa["l1_segments"]["categorical"]
cols = {c: F.LEADER_FEATURES.index(c) for c in seg_cols}
cat_sizes = {c: len(enc[c][1]) for c in seg_cols}
seg = {n: A.segment_ids(X[n]["xg"], cols, {}, cat_sizes) for n in X}
SETS, RELEASE = {}, {}
for k in run["l1_k_values"]:
    table, gmean, info = A.k_anonymous_means(seg["train"], X["train"]["cust"], X["train"]["oh"], k)
    SETS["L1_k%d" % k] = {n: np.hstack([X[n]["xg"], A.apply_segment_means(seg[n], table, gmean)]) for n in X}
    RELEASE["L1_k%d" % k] = info


def bt1_sets(seed):
    b = ad["b_t1"]
    emb = A.tree_embedding(X["train"]["xh"], b["n_trees"], b["depth"], seed)
    codes = {n: A.leaf_codes(emb, X[n]["xh"]) for n in X}
    allowed = A.k_coarsen(codes["train"], X["train"]["cust"], b["k"])
    info = {"leaves_released": int(sum(len(a) for a in allowed))}
    return {n: np.hstack([X[n]["xg"], A.one_hot_codes(codes[n], allowed)]) for n in X}, info


SETS["B_T1"], RELEASE["B_T1"] = bt1_sets(SEED0)
SETS["L0"] = {n: X[n]["xg"] for n in X}
SETS["L4"] = {n: np.hstack([X[n]["xg"], X[n]["xh"]]) for n in X}
print("各臂内地侧特征数：", {a: s["train"].shape[1] for a, s in SETS.items()})
print("k-匿名发布：", RELEASE)

编码： {'l_i_fea_1': 'freq', 'l_i_fea_2': 'freq', 'l_i_fea_4': 'freq', 'l_i_fea_5': 'freq'} 其余序号编码
{'train': ((151281, 16), (151281, 5), (151281, 23)), 'inner_val': ((59370, 16), (59370, 5), (59370, 23)), 'dev_eval': ((129489, 16), (129489, 5), (129489, 23))}


各臂内地侧特征数： {'L1_k5': 39, 'L1_k10': 39, 'L1_k20': 39, 'B_T1': 94, 'L0': 16, 'L4': 21}
k-匿名发布： {'L1_k5': {'segments_total': 177, 'segments_released': 136, 'segments_suppressed': 41}, 'L1_k10': {'segments_total': 177, 'segments_released': 117, 'segments_suppressed': 60}, 'L1_k20': {'segments_total': 177, 'segments_released': 100, 'segments_suppressed': 77}, 'B_T1': {'leaves_released': 78}}


**读法**：主动方 4 个商品类列做频数编码，其余序号编码；用户号不作特征（S6.P4 F-9）。被动方只有 5 列（独热后 23 列）。L1 的 177 个分段中，k = 10 时发布 117 个、抑制 60 个；B_T1 发布 78 个叶子。

In [4]:
configs = A.sample_configs(ad["search_space"], pre["tuning"]["configs_per_arm"], pre["seeds"]["tuning"])
TUNING = {}
t0 = time.perf_counter()
for arm in run["local_arms"]:
    best, table = A.tune(SETS[arm]["train"], X["train"]["y"], SETS[arm]["inner_val"], X["inner_val"]["y"],
                         configs, MB, SEED0, NJ)
    TUNING[arm] = {"best": best, "best_val_auc": max(r["val_auc"] for r in table), "candidates": table}
TUNING["A"] = {"best": TUNING["L4"]["best"], "source": "L4 集中式明文代理（S6.P3 freeze A-4）"}
TUNING["B_best"] = max(fa["b_best_pool"], key=lambda a: TUNING[a]["best_val_auc"])
(PRIV / run["tuning_file"]).write_text(json.dumps(TUNING, ensure_ascii=False), encoding="utf-8")
view = pd.DataFrame([dict(TUNING[a]["best"], arm=a, inner_val_auc=TUNING[a]["best_val_auc"]) for a in run["local_arms"]])
print(view.round(run["round_auc"]).to_string(index=False))
print("B_best（开发评估前按内层验证集选定）：", TUNING["B_best"], "| 调参耗时 %.0f 秒" % (time.perf_counter() - t0))

 learning_rate  max_depth  num_boost_round  reg_lambda    arm  inner_val_auc
           0.2          3              150         1.0     L0         0.6128
           0.2          4              200         1.0  L1_k5         0.6161
           0.1          6              150         1.0 L1_k10         0.6125
           0.2          3              150         1.0 L1_k20         0.6151
           0.3          3               50         1.0   B_T1         0.6335
           0.2          3              150         1.0     L4         0.6351
B_best（开发评估前按内层验证集选定）： B_T1 | 调参耗时 2741 秒


**读法**（内层验证集，20 组共享候选）：L0 0.613，三种 L1 在 0.613–0.616 之间，B_T1 0.634，集中式代理 L4 0.635。B_best 按内层验证选为 B_T1。A 的超参取自 L4（学习率 0.2、深 3、150 轮）。调参耗时受同机门禁运行影响，只作参考。

In [5]:
SCORES, COST = {}, {}
for arm in [a for a in run["local_arms"] if a != "B_T1"]:
    t0 = time.perf_counter()
    m = A.xgb_fit(SETS[arm]["train"], X["train"]["y"], TUNING[arm]["best"], MB, SEED0, NJ)
    COST[arm] = time.perf_counter() - t0
    SCORES[arm] = A.xgb_score(m, SETS[arm]["dev_eval"])
bt1 = []
for s in run["seeds"]:
    sets_s, _ = bt1_sets(s)
    m = A.xgb_fit(sets_s["train"], X["train"]["y"], TUNING["B_T1"]["best"], MB, s, NJ)
    bt1.append(A.xgb_score(m, sets_s["dev_eval"]))
SCORES["B_T1"] = np.vstack(bt1)
nc = []
for s in run["seeds"]:
    sh = {n: np.hstack([X[n]["xg"], A.shuffle_rows(X[n]["xh"], s)]) for n in ("train", "dev_eval")}
    m = A.xgb_fit(sh["train"], X["train"]["y"], TUNING["A"]["best"], MB, SEED0, NJ)
    nc.append(A.xgb_score(m, sh["dev_eval"]))
SCORES["NC"] = np.vstack(nc)
np.savez_compressed(PRIV / run["local_scores_file"], **SCORES)
print("已保存各臂开发评估集打分（私有）：", {a: np.shape(v) for a, v in SCORES.items()})
print("单次拟合耗时（秒）：", {a: round(v) for a, v in COST.items()})

已保存各臂开发评估集打分（私有）： {'L0': (129489,), 'L1_k5': (129489,), 'L1_k10': (129489,), 'L1_k20': (129489,), 'L4': (129489,), 'B_T1': (6, 129489), 'NC': (6, 129489)}
单次拟合耗时（秒）： {'L0': 22, 'L1_k5': 30, 'L1_k10': 34, 'L1_k20': 23, 'L4': 15}


**读法**：确定性臂各拟合一次；B_T1 与阴性对照各 6 个 seed。开发评估集打分只存私有目录，配对比较见 S5.G3-3。